# REM Content Studio — AI inference service on Colab

This notebook loads a real open-source instruct model (e.g. `mistralai/Mistral-7B-Instruct-v0.2`
or `meta-llama/Meta-Llama-3-8B-Instruct`) on a Colab GPU runtime and exposes it over HTTP so the
main REM Content Studio backend (running anywhere else — your laptop, a Docker container) can call
it through `LLM_BACKEND=remote_http`.

**Before running:** Runtime → Change runtime type → GPU (T4 is enough for a 4-bit 7B model).

Steps: (1) clone the repo, (2) install dependencies, (3) start the inference service in the
background, (4) open a public tunnel to it, (5) copy the printed URL into `LLM_SERVICE_URL` on
the machine running the main backend.

In [ ]:
# 1. Get the project code onto this Colab machine.
# Replace with your own fork/branch URL if different.
!git clone https://github.com/narendralama/ai-realestate.git repo
%cd repo/backend

In [ ]:
# 2. Install dependencies (base requirements + the heavy LLM ones + pyngrok for the tunnel).
!pip install -q -r requirements.txt -r requirements-llm.txt pyngrok

In [ ]:
# 3. Choose the model and 4-bit quantisation (recommended on a single T4/L4 GPU).
import os

os.environ["LLM_MODEL_NAME"] = "mistralai/Mistral-7B-Instruct-v0.2"  # or meta-llama/Meta-Llama-3-8B-Instruct
os.environ["LLM_LOAD_IN_4BIT"] = "true"

# If the model is gated on Hugging Face (Llama-3 is), log in first:
# from huggingface_hub import login
# login("hf_xxx_your_token")

In [ ]:
# 4. Start the FastAPI inference service in the background.
# This is the exact same ai_service/service_app.py used for local dev — it loads the
# model once on startup and exposes POST /generate.
get_ipython().system_raw(
    "LLM_MODEL_NAME=$LLM_MODEL_NAME LLM_LOAD_IN_4BIT=$LLM_LOAD_IN_4BIT "
    "uvicorn ai_service.service_app:app --host 0.0.0.0 --port 9000 > service.log 2>&1 &"
)

import time
time.sleep(10)
!tail -n 20 service.log

In [ ]:
# 5. Wait for the model to finish loading (can take a few minutes for a 7B model),
# then check /health repeatedly until status is "ok".
import time
import urllib.request
import json

for _ in range(60):
    try:
        with urllib.request.urlopen("http://localhost:9000/health", timeout=5) as resp:
            status = json.load(resp)
        print(status)
        if status.get("status") == "ok":
            break
    except Exception as exc:
        print("not up yet:", exc)
    time.sleep(10)

In [ ]:
# 6. Open a public tunnel to port 9000 so the outside world (your laptop / Docker backend) can reach it.
# Sign up for a free ngrok account and paste your authtoken below: https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok

ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")
public_url = ngrok.connect(9000, "http")
print("Public URL:", public_url)
print("\nOn the machine running the main backend, set:")
print(f"  LLM_BACKEND=remote_http")
print(f"  LLM_SERVICE_URL={public_url}")

In [ ]:
# 7. Optional: sanity-check the endpoint directly from this notebook before wiring up the backend.
import requests

resp = requests.post(
    "http://localhost:9000/generate",
    json={"prompt": "Write a one-sentence real estate listing for a 3-bedroom house in Richmond.", "max_new_tokens": 80},
    timeout=60,
)
print(resp.json())